In [ ]:
pip install langchain langchain-core langchain-openai openai langchain-anthropic langchain-google-genai google-generativeai langchain-huggingface transformers huggingface-hub python-dotenv numpy scikit-learn

#**Output as json**
> if the model supports you can use ***with_structured_output*** function else you need to use **output parsers**

##with_structured_output

> *typed dictionary, pydantic, json_schema*

###typed dictionary

In [ ]:
#typed dictionary
'''it dosent validate during runtime but tells what type key should be while writing code'''

class Person:
  name:str
  id:int

d: Person = {"name": "hello", "id": 10}

In [ ]:
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from google.colab import userdata
from typing import TypedDict
huggingface_api_key = userdata.get('HUGGINGFACEHUB_ACCESS_TOKEN')

class Person(TypedDict):
  name:str
  topic: str

llm = HuggingFaceEndpoint(huggingfacehub_api_token=huggingface_api_key, repo_id="deepseek-ai/DeepSeek-V4-Flash", task="text-generation")
model = ChatHuggingFace(llm=llm)
new_model = model.with_structured_output(Person) # mode = json to get output as json, mode = function calling if you want to call a function

result = new_model.invoke("Bharat is from class 10 with poor marks in his CBSE marks due to mismanagement of the exam by the board that conducts it.")
print(result)

{'name': 'Bharat', 'topic': 'CBSE exam mismanagement and poor marks'}


In [ ]:
# if model dosent understand context of output to be gen
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from google.colab import userdata
from typing import TypedDict, Annotated, Optional, Literal
huggingface_api_key = userdata.get('HUGGINGFACEHUB_ACCESS_TOKEN')

class Person(TypedDict):
  name: Annotated[str, "name of the main character"]
  topic: Annotated[str, "One word that describes the topic the person is involved in"]
  wrong: Annotated[Optional[str], "what went wrong in his situation in atmost 3 words"]
  outcome: Annotated[Literal["worst", "ok", "great"], "what was the outcome to that person"]

llm = HuggingFaceEndpoint(huggingfacehub_api_token=huggingface_api_key, repo_id="deepseek-ai/DeepSeek-V4-Flash", task="text-generation")
model = ChatHuggingFace(llm=llm)
new_model = model.with_structured_output(Person)

result = new_model.invoke("Bharat is from class 10 with poor marks in his CBSE marks due to mismanagement of the exam by the board that conducts it due to which he sucided.")
print(result)

{'name': 'Bharat', 'topic': 'Exams', 'wrong': 'Exam mismanagement', 'outcome': 'worst'}


###Pydantic

In [ ]:
#pydantic not only imposes types, but also validates it
!pip install pydantic email_validator

In [ ]:
from pydantic import BaseModel, EmailStr, Field

class Person(BaseModel):
  name:str #for things like optional, literal, annotated you can just use typing module things
  name_:str = "default_value"
  email: EmailStr = "abc@gmail.com"
  marks: float = Field(gt=0, lt=10, default=5, description="marks")

student = {"name": "bharat", "marks": 5}

print(Person(**student))

name='bharat' name_='default_value' email='abc@gmail.com' marks=5.0


###JSON

In [ ]:
#use json if you are working with multiple languages
'''format of json document'''

{
    "title": "person",
    "description": "schema of person",
    "type": "object",
    "properties": {
        "name": "string",
        "age": "integer"
    },
    "required": ["name"]
}

In [ ]:
#An example json schema
json_schema = {
  "title": "Review",
  "type": "object",
  "properties": {
    "key_themes": {
      "type": "array",
      "items": {
        "type": "string"
      },
      "description": "Write down all the key themes discussed in the review in a list"
    },
    "summary": {
      "type": "string",
      "description": "A brief summary of the review"
    },
    "sentiment": {
      "type": "string",
      "enum": ["pos", "neg"],
      "description": "Return sentiment of the review either negative, positive or neutral"
    },
    "pros": {
      "type": ["array", "null"],
      "items": {
        "type": "string"
      },
      "description": "Write down all the pros inside a list"
    },
    "cons": {
      "type": ["array", "null"],
      "items": {
        "type": "string"
      },
      "description": "Write down all the cons inside a list"
    },
    "name": {
      "type": ["string", "null"],
      "description": "Write the name of the reviewer"
    }
  },
  "required": ["key_themes", "summary", "sentiment"]
}

Everything else is same for all 3 from with standard output

##Output parsers

> ***StrOutputParser*** - For string output, ***JSONOutputParser*** - Json output - But not structured, ***StructuredOutputParser*** - Structured Json - But No validation, ***PydanticOutputParser*** - Structured Json and Validated

###StrOutputParser

In [ ]:
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from langchain_core.prompts import PromptTemplate, MessagesPlaceholder
from langchain_core.output_parsers import StrOutputParser
from google.colab import userdata
huggingface_api_key = userdata.get('HUGGINGFACEHUB_ACCESS_TOKEN')

template1 = PromptTemplate(template="write a detailed report on {topic}", input_variables=["topic"])

template2 = PromptTemplate(template="summarise this text {text}", input_variables=["text"])

llm = HuggingFaceEndpoint(huggingfacehub_api_token = huggingface_api_key, repo_id="meta-llama/Llama-3.1-8B-Instruct", task = "text-generation")
model = ChatHuggingFace(llm=llm)

parser = StrOutputParser()
chain = template1 | model | parser | template2 | model | parser
result = chain.invoke({
    "topic": "AI"
})

print(result)

### **Summary of the Comprehensive Report on Artificial Intelligence (AI)**

#### **1. Introduction**
AI is a transformative technology that enables machines to simulate human intelligence, impacting industries, economies, and daily life. This report covers AI's fundamentals, applications, ethical concerns, challenges, and future trends.

#### **2. Definition & Types of AI**
- **AI** refers to computer systems performing tasks like learning, reasoning, perception, decision-making, and natural language processing (NLP).
- **Types of AI**:
  - **Narrow (Weak) AI**: Task-specific (e.g., Siri, chatbots).
  - **General (Strong) AI**: Human-like reasoning (not yet achieved).
  - **Superintelligent AI**: Hypothetical AI surpassing human intelligence.

#### **3. Key AI Technologies**
- **Machine Learning (ML)**: Algorithms that learn from data.
- **Natural Language Processing (NLP)**: Enables AI to understand and generate human language.
- **Computer Vision**: AI interpreting visual data (e.g.

###JsonOutputParser

In [ ]:
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import JsonOutputParser
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from google.colab import userdata
huggingface_api_key = userdata.get('HUGGINGFACEHUB_ACCESS_TOKEN')

parser = JsonOutputParser()
template = PromptTemplate(template="write a summary on blackhole {format_instruction}", partial_variables={
    "format_instruction": parser.get_format_instructions()
})

llm = HuggingFaceEndpoint(huggingfacehub_api_token = huggingface_api_key, repo_id="meta-llama/Llama-3.1-8B-Instruct", task = "text-generation")
model = ChatHuggingFace(llm=llm)

'''instead of this'''
# prompt = template.format()
# result = model.invoke(prompt)
# print(parser.parse(result.content))

'''you can use this'''
chain = template | model | parser
result = chain.invoke({})
print(result)

{'summary': 'A black hole is a region in spacetime where gravity is so strong that nothing, not even light, can escape. They form when massive stars collapse under their own gravity at the end of their life cycle. Black holes have an event horizon, a boundary beyond which escape is impossible, and a singularity at their center where matter is compressed to infinite density. They play a crucial role in astrophysics, influencing galaxy formation, and their existence has been confirmed through observations of gravitational waves and the Event Horizon Telescope.', 'key_points': ['Formed from the collapse of massive stars', 'Infinite gravitational pull beyond the event horizon', 'Singularity at the center', 'Can be detected through gravitational waves and radiation', 'Supermassive black holes exist at the centers of galaxies']}


###StructuredOutputParser

In [3]:
!pip install langchain_classic

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 14.0 MB/s eta 0:00:00


In [22]:
from langchain_classic.output_parsers import StructuredOutputParser, ResponseSchema
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from langchain_core.prompts import PromptTemplate
from google.colab import userdata

huggingface_api_key = userdata.get('HUGGINGFACEHUB_ACCESS_TOKEN')

llm = HuggingFaceEndpoint(huggingfacehub_api_token = huggingface_api_key, repo_id="meta-llama/Llama-3.1-8B-Instruct", task = "text-generation")
model = ChatHuggingFace(llm=llm)

schema = [
    ResponseSchema(name = 'fact_1', description='fact 1 of topic'),
    ResponseSchema(name = 'fact_2', description='fact 2 of topic'),
    ResponseSchema(name = 'fact_3', description='fact 3 of topic')
]

parser = StructuredOutputParser.from_response_schemas(schema)
format_instructions = parser.get_format_instructions()

template = PromptTemplate(template="give 3 facts on {topic} \n {format_instructions}", input_variables=["topic"], partial_variables={
    "format_instructions": format_instructions
})

'''you can use this'''
# chain = template | model | parser
# print(chain.invoke("Hi"))

'''or this'''
prompt = template.invoke({
    "topic": "Hi - Hello"
    })
result = model.invoke(prompt)
print(parser.parse(result.content))

{'fact_1': "The word 'Hello' is one of the most widely recognized greetings in the world, used by over 80% of the global population.", 'fact_2': "The first recorded use of the word 'Hello' as a greeting dates back to 1839 in the United States, where it was first used as a telephone greeting.", 'fact_3': "In some cultures, saying 'Hello' is considered impolite or even rude, such as in Japan, where it's considered more polite to use a more formal greeting like 'Konnichiwa'."}


###PydanticOutputParser

In [37]:
from langchain_core.output_parsers import PydanticOutputParser
from langchain_huggingface import ChatHuggingFace, HuggingFaceEndpoint
from langchain_core.prompts import PromptTemplate
from pydantic import BaseModel, Field
from google.colab import userdata

huggingface_api_key = userdata.get('HUGGINGFACEHUB_ACCESS_TOKEN')

llm = HuggingFaceEndpoint(huggingfacehub_api_token = huggingface_api_key, repo_id="meta-llama/Llama-3.1-8B-Instruct", task = "text-generation")
model = ChatHuggingFace(llm=llm)


class Person(BaseModel):
  name:str = Field(description="name of the person")
  age:int = Field(gt=18, description="age of the person")
  city:str = Field(description="city from which the person belongs to")

parser = PydanticOutputParser(pydantic_object=Person)

# it gave errors without mentioning to return json, small models hellusinate and it is giving me python code to generate json, so I mentioned it to only return json
template = PromptTemplate(template="Generate the name, age and city of a ficitonal {place} person Return ONLY a JSON object. \n {format_instructions}", input_variables=["place"], partial_variables={
    "format_instructions": parser.get_format_instructions()
})
'''you can also use this'''
# chain = template | model | parser
# print(chain.invoke({
#     "place": "indian"
# }))

prompt = template.invoke({
    "place": "indian"
})

result = model.invoke(prompt)
print(parser.parse(result.content))

name='Rohan Sharma' age=32 city='Mumbai'
